In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_batch = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(
        "abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Bronze.Lakehouse/Files/bronze/incremental/batch_02/seo_traffic/seo_traffic_batch_02.csv"
    )
)

print("Batch 02 source rows:", df_batch.count())

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 3, Finished, Available, Finished, False)

Batch 02 source rows: 16672


In [2]:
gold_watermark_path = "abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Gold.Lakehouse/Tables/dbo/control_watermark"

df_watermark = (
    spark.read
    .format("delta")
    .load(gold_watermark_path)
)

traffic_watermark = (
    df_watermark
    .filter(F.col("table_name") == "Fact_SEO_Traffic")
    .select("last_watermark")
    .first()["last_watermark"]
)

print("Previous Traffic watermark:", traffic_watermark)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 4, Finished, Available, Finished, False)

Previous Traffic watermark: 2026-01-31 23:54:00


In [3]:
df_incremental = df_batch.filter(
    F.col("updated_timestamp") > F.lit(traffic_watermark)
)

print("Incremental rows:", df_incremental.count())

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 5, Finished, Available, Finished, False)

Incremental rows: 16672


In [4]:
df_incremental.select(
    F.min("updated_timestamp").alias("min_updated_timestamp"),
    F.max("updated_timestamp").alias("max_updated_timestamp")
).show(truncate=False)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 6, Finished, Available, Finished, False)

+---------------------+---------------------+
|min_updated_timestamp|max_updated_timestamp|
+---------------------+---------------------+
|2026-02-01 00:05:00  |2026-05-31 23:27:00  |
+---------------------+---------------------+



In [5]:
df_clean = (
    df_incremental
    .withColumn(
        "country_code",
        F.when(
            F.upper(F.trim(F.col("country_code"))).isin(
                "USA", "UNITED STATES", "US"
            ),
            F.lit("US")
        )
        .otherwise(F.upper(F.trim(F.col("country_code"))))
    )
    .withColumn(
        "device_type",
        F.initcap(F.trim(F.col("device_type")))
    )
)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 7, Finished, Available, Finished, False)

In [6]:
df_clean = df_clean.filter(
    (F.col("visits") >= 0) &
    (F.col("revenue") >= 0) &
    (F.col("clicks") >= 0) &
    (F.col("impressions") >= 0) &
    (F.col("orders") >= 0)
)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 8, Finished, Available, Finished, False)

In [7]:
window_spec = Window.partitionBy(
    "traffic_id"
).orderBy(
    F.col("updated_timestamp").desc()
)

df_clean = (
    df_clean
    .withColumn("_rn", F.row_number().over(window_spec))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print("Clean incremental rows:", df_clean.count())

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 9, Finished, Available, Finished, False)

Clean incremental rows: 16439


In [8]:
df_invalid = df_incremental.filter(
    (F.col("visits") < 0) |
    (F.col("revenue") < 0) |
    (F.col("clicks") < 0) |
    (F.col("impressions") < 0) |
    (F.col("orders") < 0)
)

print("Invalid records:", df_invalid.count())

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 10, Finished, Available, Finished, False)

Invalid records: 1


In [9]:
df_invalid.select(
    F.sum(F.when(F.col("visits") < 0, 1).otherwise(0)).alias("negative_visits"),
    F.sum(F.when(F.col("revenue") < 0, 1).otherwise(0)).alias("negative_revenue"),
    F.sum(F.when(F.col("clicks") < 0, 1).otherwise(0)).alias("negative_clicks"),
    F.sum(F.when(F.col("impressions") < 0, 1).otherwise(0)).alias("negative_impressions"),
    F.sum(F.when(F.col("orders") < 0, 1).otherwise(0)).alias("negative_orders")
).show()

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 11, Finished, Available, Finished, False)

+---------------+----------------+---------------+--------------------+---------------+
|negative_visits|negative_revenue|negative_clicks|negative_impressions|negative_orders|
+---------------+----------------+---------------+--------------------+---------------+
|              1|               0|              0|                   0|              0|
+---------------+----------------+---------------+--------------------+---------------+



In [10]:
print("Before cleaning:", df_incremental.count())
print("After cleaning:", df_clean.count())

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 12, Finished, Available, Finished, False)

Before cleaning: 16672
After cleaning: 16439


In [11]:
print(
    "Current Silver Traffic rows:",
    spark.read.table("fact_seo_traffic").count()
)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 13, Finished, Available, Finished, False)

Current Silver Traffic rows: 49995


In [12]:
df_clean.createOrReplaceTempView("traffic_incremental")

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 14, Finished, Available, Finished, False)

In [13]:
spark.sql("""
MERGE INTO fact_seo_traffic AS target
USING traffic_incremental AS source
ON target.traffic_id = source.traffic_id

WHEN MATCHED
     AND source.updated_timestamp > target.updated_timestamp
THEN UPDATE SET *

WHEN NOT MATCHED
THEN INSERT *
""")

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 15, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [14]:
df_silver = spark.read.table("fact_seo_traffic")

print("Silver Traffic rows:", df_silver.count())

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 16, Finished, Available, Finished, False)

Silver Traffic rows: 49995


In [15]:
df_silver.groupBy("traffic_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 17, Finished, Available, Finished, False)

+----------+-----+
|traffic_id|count|
+----------+-----+
+----------+-----+



In [16]:
df_silver.select(
    F.min("updated_timestamp").alias("min_updated_timestamp"),
    F.max("updated_timestamp").alias("max_updated_timestamp")
).show(truncate=False)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 18, Finished, Available, Finished, False)

+---------------------+---------------------+
|min_updated_timestamp|max_updated_timestamp|
+---------------------+---------------------+
|2025-10-01 00:02:00  |2026-09-30 23:46:00  |
+---------------------+---------------------+



In [17]:
print(
    "Silver Traffic rows:",
    spark.read.table("fact_seo_traffic").count()
)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 19, Finished, Available, Finished, False)

Silver Traffic rows: 49995


In [18]:
df_silver = spark.read.table("fact_seo_traffic")

df_silver.groupBy("traffic_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 20, Finished, Available, Finished, False)

+----------+-----+
|traffic_id|count|
+----------+-----+
+----------+-----+



In [19]:
df_silver.select(
    F.min("updated_timestamp").alias("min_updated_timestamp"),
    F.max("updated_timestamp").alias("max_updated_timestamp")
).show(truncate=False)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 21, Finished, Available, Finished, False)

+---------------------+---------------------+
|min_updated_timestamp|max_updated_timestamp|
+---------------------+---------------------+
|2025-10-01 00:02:00  |2026-09-30 23:46:00  |
+---------------------+---------------------+



In [20]:
df_silver = spark.read.table("fact_seo_traffic")

print("Silver Traffic rows:", df_silver.count())

duplicate_count = (
    df_silver.groupBy("traffic_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate traffic IDs:", duplicate_count)

df_silver.select(
    F.min("updated_timestamp").alias("min_updated_timestamp"),
    F.max("updated_timestamp").alias("max_updated_timestamp")
).show(truncate=False)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 22, Finished, Available, Finished, False)

Silver Traffic rows: 49995
Duplicate traffic IDs: 0
+---------------------+---------------------+
|min_updated_timestamp|max_updated_timestamp|
+---------------------+---------------------+
|2025-10-01 00:02:00  |2026-09-30 23:46:00  |
+---------------------+---------------------+



In [21]:
gold_watermark_path = "abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Gold.Lakehouse/Tables/dbo/control_watermark"

df_watermark = (
    spark.read
    .format("delta")
    .load(gold_watermark_path)
)

df_watermark.show(truncate=False)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 23, Finished, Available, Finished, False)

+-----------+------------------------+-------------------+
|source_name|table_name              |last_watermark     |
+-----------+------------------------+-------------------+
|SEO        |Fact_Keyword_Performance|2026-01-31 23:50:00|
|SEO        |Fact_SEO_Traffic        |2026-01-31 23:54:00|
|SEO        |Fact_SEO_Orders         |2026-01-31 22:38:00|
+-----------+------------------------+-------------------+



In [22]:
df_watermark_updated = (
    df_watermark
    .withColumn(
        "last_watermark",
        F.when(
            F.col("table_name") == "Fact_SEO_Traffic",
            F.lit("2026-05-31 23:27:00").cast("timestamp")
        )
        .otherwise(F.col("last_watermark"))
    )
)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 24, Finished, Available, Finished, False)

In [23]:
df_watermark_updated.show(truncate=False)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 25, Finished, Available, Finished, False)

+-----------+------------------------+-------------------+
|source_name|table_name              |last_watermark     |
+-----------+------------------------+-------------------+
|SEO        |Fact_Keyword_Performance|2026-01-31 23:50:00|
|SEO        |Fact_SEO_Traffic        |2026-05-31 23:27:00|
|SEO        |Fact_SEO_Orders         |2026-01-31 22:38:00|
+-----------+------------------------+-------------------+



In [24]:
(
    df_watermark_updated.write
    .format("delta")
    .mode("overwrite")
    .save(gold_watermark_path)
)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 26, Finished, Available, Finished, False)

In [25]:
spark.read.format("delta").load(
    gold_watermark_path
).show(truncate=False)

StatementMeta(, 18e00ac2-db79-463a-bb7d-5bb976c69210, 27, Finished, Available, Finished, False)

+-----------+------------------------+-------------------+
|source_name|table_name              |last_watermark     |
+-----------+------------------------+-------------------+
|SEO        |Fact_Keyword_Performance|2026-01-31 23:50:00|
|SEO        |Fact_SEO_Traffic        |2026-05-31 23:27:00|
|SEO        |Fact_SEO_Orders         |2026-01-31 22:38:00|
+-----------+------------------------+-------------------+

